# RNA-seq Differential Expression Analysis
## TL;DR — Plain English Introduction

**What you will learn:** How to go from a gene expression count matrix to a list of differentially expressed genes, complete with statistical testing, multiple testing correction, and biological visualization.

**Why it matters:** RNA-seq is the most widely used technique in modern molecular biology. If you work in pharmaceutical ML, clinical genomics, or any biology-adjacent field, you will encounter count matrices.

**Real dataset used:** We simulate a realistic count matrix mirroring a cancer vs. normal comparison, then walk through the same pipeline used with real GEO datasets.


## Learning Objectives

By the end of this notebook you will be able to:
1. Understand why raw RNA-seq counts need normalization (CPM, TPM, DESeq2 size factors)
2. Identify differentially expressed genes with statistical rigor (negative binomial model)
3. Apply multiple testing correction (Benjamini-Hochberg FDR)
4. Create a volcano plot and heatmap from DE results
5. Recognize when batch effects confound your analysis (and how to detect them)


## 🟢 Complete Beginner's Guide

**Assumed background:** Zero biology. Zero bioinformatics.

### What you need to know before starting

- **Gene** — a stretch of DNA that encodes instructions (usually for making a protein).
- **mRNA (messenger RNA)** — a temporary copy of a gene that carries instructions from the nucleus to the cell's protein-making machinery.
- **Gene expression** — how much a gene is 'turned on' in a cell. High expression = many mRNA copies made.
- **RNA-seq** — a technology that counts mRNA molecules in a sample, giving us gene expression levels.
- **Counts matrix** — a table where rows are genes, columns are samples (e.g. patients), and each value is how many mRNA molecules were found for that gene in that sample.
- **Differential expression** — comparing gene expression between two conditions (e.g. healthy vs diseased) to find genes that change.

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `transcript` | The mRNA molecule produced from one gene |
| `read` | A short DNA/RNA sequence measured by the sequencer |
| `normalization` | Adjusting counts so samples with more total reads are comparable |
| `TPM / RPKM / FPKM` | Common normalization units for RNA-seq data |
| `log2 fold change` | How much a gene's expression doubled or halved between conditions |
| `p-value / FDR` | Statistical confidence that an expression change is real, not random |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** — understand the biological question before looking at code.
2. **Run code cells one at a time** — inspect the shape of each dataframe with `.head()` and `.shape`.
3. **Modify one number and re-run** — change a threshold (e.g. fold-change cutoff from 2 to 1.5) and observe how many genes are affected.

### If you're stuck

- **YouTube:** 'StatQuest: RNA-seq results explained' by Josh Starmer (https://www.youtube.com/watch?v=tlf6wYJrwKY) — best plain-English intro.
- **YouTube:** 'A gentle introduction to RNA-seq' by Bioinformatics Coach (https://www.youtube.com/watch?v=tlf6wYJrwKY)
- **Book:** *Bioinformatics Data Skills* by Vince Buffalo, Chapter 8 (RNA-seq pipeline overview).
- **Web:** https://hbctraining.github.io/Intro-to-R-RNAseq/ — Harvard free RNA-seq course.


## 🎯 Predict Before Running

1. You have two samples: sample A has 10 million reads, sample B has 20 million reads. Gene X has 100 reads in A and 150 reads in B. Is Gene X upregulated in B? Why can't you tell from raw counts alone?
2. You test 20,000 genes at α=0.05. How many false positives do you expect by chance if the null is true for all genes?
3. Why does RNA-seq use a Negative Binomial distribution rather than Poisson for modeling counts?


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from scipy.stats import nbinom
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── SIMULATE REALISTIC RNA-SEQ COUNT DATA ──
n_genes = 5000
n_samples_per_group = 4  # 4 tumor, 4 normal

# Gene parameters: mean expression and dispersion (overdispersion)
# Most genes are lowly expressed; few are highly expressed (negative binomial)
mean_expression = np.random.lognormal(mean=3, sigma=2, size=n_genes)  # per million
dispersion = 0.5 / np.sqrt(mean_expression) + 0.1  # typical RNA-seq dispersion

# Library sizes (realistic variation: 10-30M reads per sample)
lib_sizes_tumor  = np.random.randint(10_000_000, 30_000_000, n_samples_per_group)
lib_sizes_normal = np.random.randint(10_000_000, 30_000_000, n_samples_per_group)

# Generate counts: Normal group
counts_normal = np.zeros((n_genes, n_samples_per_group), dtype=int)
for s in range(n_samples_per_group):
    mu = mean_expression * lib_sizes_normal[s] / 1e6
    for g in range(n_genes):
        p = 1 / (1 + mu[g] * dispersion[g])
        r = 1 / dispersion[g]
        counts_normal[g, s] = nbinom.rvs(r, p)

# Generate counts: Tumor group (500 genes truly DE)
de_genes = np.random.choice(n_genes, 500, replace=False)
fold_changes = np.ones(n_genes)
fold_changes[de_genes[:250]] = np.random.uniform(2, 8, 250)   # upregulated
fold_changes[de_genes[250:]] = np.random.uniform(0.1, 0.4, 250)  # downregulated

counts_tumor = np.zeros((n_genes, n_samples_per_group), dtype=int)
for s in range(n_samples_per_group):
    mu = mean_expression * fold_changes * lib_sizes_tumor[s] / 1e6
    for g in range(n_genes):
        p = 1 / (1 + mu[g] * dispersion[g])
        r = 1 / dispersion[g]
        counts_tumor[g, s] = nbinom.rvs(r, p)

# Assemble DataFrame
sample_names = [f'Tumor_{i+1}' for i in range(n_samples_per_group)] +                [f'Normal_{i+1}' for i in range(n_samples_per_group)]
gene_names = [f'GENE_{i:05d}' for i in range(n_genes)]
counts = pd.DataFrame(
    np.hstack([counts_tumor, counts_normal]),
    index=gene_names, columns=sample_names
)

print("COUNT MATRIX SUMMARY:")
print(f"  Shape: {counts.shape} (genes × samples)")
print(f"  Library sizes: {counts.sum().values}")
print(f"  Median counts per gene: {counts.median(axis=1).median():.1f}")
print(f"  Zero counts: {(counts==0).sum().sum()} / {counts.size} ({(counts==0).mean().mean()*100:.1f}%)")
print(f"  True DE genes injected: 500")
print()
print("First 3 genes × first 4 samples:")
print(counts.iloc[:3, :4])

In [ ]:
# ── NORMALIZATION ──
# Problem: raw counts are proportional to library size
# Must normalize before comparison

# Method 1: CPM (Counts Per Million) — simple, widely used
def cpm(count_matrix):
    lib_sizes = count_matrix.sum(axis=0)
    return count_matrix * 1e6 / lib_sizes

# Method 2: Size-factor normalization (simplified DESeq2 approach)
# DESeq2's median-of-ratios method
def size_factor_normalize(count_matrix):
    """Simplified DESeq2 size factor estimation."""
    # Remove genes with any zero (geometric mean undefined)
    nonzero = count_matrix[(count_matrix > 0).all(axis=1)]
    if len(nonzero) == 0:
        return count_matrix / count_matrix.sum(axis=0) * count_matrix.sum(axis=0).mean()

    # Geometric mean of each gene across samples
    log_counts = np.log(nonzero + 0.5)
    geo_means = log_counts.mean(axis=1)

    # For each sample, compute median ratio to geometric mean
    size_factors = np.zeros(count_matrix.shape[1])
    for s in range(count_matrix.shape[1]):
        ratios = log_counts.iloc[:, s] - geo_means
        size_factors[s] = np.exp(np.median(ratios))

    return count_matrix / size_factors

counts_cpm = cpm(counts)
counts_sf = size_factor_normalize(counts)

print("NORMALIZATION COMPARISON:")
print("Before normalization — library sizes:")
print(f"  {counts.sum().values}")
print()
print("After CPM normalization — all sum to 1,000,000:")
print(f"  {counts_cpm.sum().round().values}")
print()
print("Size factor estimates (DESeq2-like):")
lib_ratios = counts.sum() / counts.sum().mean()
print(f"  Raw library ratios: {lib_ratios.round(3).values}")

print()
print("ANSWER TO PREDICT Q1:")
print("  Sample A (10M reads): 100 reads → CPM = 10.0")
print("  Sample B (20M reads): 150 reads → CPM = 7.5")
print("  Gene X is actually DOWNREGULATED in B after normalization!")
print("  Raw count comparison is meaningless without normalizing for library size.")

In [ ]:
# ── DIFFERENTIAL EXPRESSION TESTING ──
# We use Welch's t-test on log-CPM values (simplified; production uses DESeq2/edgeR)

from scipy.stats import ttest_ind

# Filter lowly expressed genes (< 1 CPM in at least 3 samples)
keep = (counts_cpm >= 1).sum(axis=1) >= 3
counts_filt = counts[keep]
counts_cpm_filt = counts_cpm[keep]
print(f"After low-expression filtering: {keep.sum()} / {n_genes} genes retained")

# Log-transform (log2 CPM, adding pseudocount)
log_cpm = np.log2(counts_cpm_filt + 1)

tumor_cols  = [c for c in counts.columns if 'Tumor' in c]
normal_cols = [c for c in counts.columns if 'Normal' in c]

# Test each gene
results = []
for gene in log_cpm.index:
    t, normal = log_cpm.loc[gene, tumor_cols].values, log_cpm.loc[gene, normal_cols].values
    stat, pval = ttest_ind(t, normal, equal_var=False)
    log2fc = t.mean() - normal.mean()
    results.append({'gene': gene, 'log2FC': log2fc, 'pvalue': pval, 'baseMean': log_cpm.loc[gene].mean()})

results_df = pd.DataFrame(results)

# Multiple testing correction: Benjamini-Hochberg FDR
def bh_correction(pvalues, alpha=0.05):
    """Benjamini-Hochberg FDR correction."""
    n = len(pvalues)
    order = np.argsort(pvalues)
    sorted_p = pvalues[order]
    adjusted = np.zeros(n)
    for i in range(n-1, -1, -1):
        if i == n-1:
            adjusted[order[i]] = sorted_p[i]
        else:
            adjusted[order[i]] = min(adjusted[order[i+1]], sorted_p[i] * n / (i+1))
    return np.clip(adjusted, 0, 1)

results_df['padj'] = bh_correction(results_df['pvalue'].values)
results_df['significant'] = (results_df['padj'] < 0.05) & (results_df['log2FC'].abs() > 1)

# Summary
n_sig = results_df['significant'].sum()
n_up = ((results_df['padj'] < 0.05) & (results_df['log2FC'] > 1)).sum()
n_dn = ((results_df['padj'] < 0.05) & (results_df['log2FC'] < -1)).sum()

print(f"\nDIFFERENTIAL EXPRESSION RESULTS:")
print(f"  Genes tested: {len(results_df)}")
print(f"  Significant (padj<0.05, |log2FC|>1): {n_sig}")
print(f"  Upregulated in tumor: {n_up}")
print(f"  Downregulated in tumor: {n_dn}")

print(f"\nANSWER TO PREDICT Q2:")
print(f"  At α=0.05 × 20,000 genes = {0.05*20000:.0f} expected false positives!")
print(f"  BH correction controls FDR: {n_sig} significant at FDR<5%")
print(f"  (Expected FP under BH: {n_sig * 0.05:.1f})")

print(f"\nANSWER TO PREDICT Q3:")
print("  Poisson: mean = variance (too restrictive for gene expression)")
print("  Negative Binomial: variance = mean + mean²/r (overdispersion parameter r)")
print("  Real RNA-seq data always shows overdispersion: biological + technical noise")

In [ ]:
# ── VISUALIZATION: VOLCANO PLOT ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Volcano plot
ax = axes[0]
colors = np.where(results_df['significant'] & (results_df['log2FC'] > 0), '#d62728',
         np.where(results_df['significant'] & (results_df['log2FC'] < 0), '#1f77b4', '#888888'))

ax.scatter(results_df['log2FC'], -np.log10(results_df['pvalue'] + 1e-300),
           c=colors, alpha=0.4, s=8, rasterized=True)
ax.axvline(-1, color='gray', linestyle='--', alpha=0.5)
ax.axvline(+1, color='gray', linestyle='--', alpha=0.5)
ax.axhline(-np.log10(0.05), color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('log₂ Fold Change (Tumor / Normal)')
ax.set_ylabel('-log₁₀(p-value)')
ax.set_title(f'Volcano Plot\n{n_up} up (red), {n_dn} down (blue), {n_sig} total significant')

# MA plot (mean vs fold-change — better for RNA-seq)
ax2 = axes[1]
ax2.scatter(results_df['baseMean'], results_df['log2FC'],
            c=colors, alpha=0.4, s=8, rasterized=True)
ax2.axhline(0, color='black', linewidth=1)
ax2.axhline(1, color='gray', linestyle='--', alpha=0.5)
ax2.axhline(-1, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Mean log₂ Expression')
ax2.set_ylabel('log₂ Fold Change')
ax2.set_title('MA Plot\n(heteroskedasticity: high-expr genes more stable)')

plt.tight_layout()
plt.savefig('rnaseq_volcano.png', dpi=100)
plt.show()

# Top DE genes
top_up = results_df[results_df['significant'] & (results_df['log2FC']>0)].nlargest(5, 'log2FC')
top_dn = results_df[results_df['significant'] & (results_df['log2FC']<0)].nsmallest(5, 'log2FC')
print("Top 5 upregulated genes:")
print(top_up[['gene','log2FC','padj']].to_string(index=False))
print("\nTop 5 downregulated genes:")
print(top_dn[['gene','log2FC','padj']].to_string(index=False))

In [ ]:
# ── BATCH EFFECT DETECTION AND CORRECTION ──
# Critical skill: recognizing when technical variation (batch) dominates biological signal

# Simulate data with batch effect (samples from two different labs)
np.random.seed(99)
n_g = 200

# True signal: 50 genes are DE between conditions
true_log2fc = np.zeros(n_g)
true_log2fc[:50] = np.random.uniform(1.5, 3, 50)  # upregulated in condition 2

# Batch effect: 100 genes have batch-specific expression shifts
batch_effect = np.zeros(n_g)
batch_effect[50:150] = np.random.normal(2, 0.5, 100)  # batch 2 artificially high

# 4 samples per group: 2 from each batch in each condition
# Batch 1: samples 1,2 (condition 1) + samples 5,6 (condition 2)
# Batch 2: samples 3,4 (condition 1) + samples 7,8 (condition 2)
data = np.zeros((n_g, 8))
for i, (cond_fc, batch_fc, noise_scale) in enumerate([
    (0, 0, 0.3), (0, 0, 0.3),          # Batch1, Cond1
    (0, batch_effect, 0.3), (0, batch_effect, 0.3),  # Batch2, Cond1
    (true_log2fc, 0, 0.3), (true_log2fc, 0, 0.3),     # Batch1, Cond2
    (true_log2fc, batch_effect, 0.3), (true_log2fc, batch_effect, 0.3), # Batch2, Cond2
]):
    mean_expr = np.ones(n_g) * 5 + cond_fc + batch_fc
    data[:, i] = mean_expr + np.random.normal(0, noise_scale, n_g)

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

labels_cond  = ['Cond1']*4 + ['Cond2']*4
labels_batch = ['Batch1','Batch1','Batch2','Batch2']*2

# PCA before batch correction
scaler = StandardScaler()
pca = PCA(n_components=2)
pcs = pca.fit_transform(scaler.fit_transform(data.T))

# Simple batch correction: subtract batch mean (simplified ComBat idea)
data_corrected = data.copy()
batch1_idx = [0, 1, 4, 5]
batch2_idx = [2, 3, 6, 7]
batch_mean_diff = data[:, batch2_idx].mean(axis=1) - data[:, batch1_idx].mean(axis=1)
for i in batch2_idx:
    data_corrected[:, i] -= batch_mean_diff

pcs_corr = pca.fit_transform(scaler.fit_transform(data_corrected.T))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_cond  = ['#1f77b4' if c=='Cond1' else '#d62728' for c in labels_cond]
markers_batch = ['o' if b=='Batch1' else 's' for b in labels_batch]

for ax, pcs_data, title in [(axes[0], pcs, 'BEFORE Batch Correction'),
                             (axes[1], pcs_corr, 'AFTER Batch Correction')]:
    for i in range(8):
        ax.scatter(pcs_data[i,0], pcs_data[i,1], c=colors_cond[i],
                   marker=markers_batch[i], s=120, zorder=5)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(title)
    from matplotlib.lines import Line2D
    legend = [Line2D([0],[0], marker='o', color='w', markerfacecolor='#1f77b4', markersize=10, label='Cond1'),
              Line2D([0],[0], marker='o', color='w', markerfacecolor='#d62728', markersize=10, label='Cond2'),
              Line2D([0],[0], marker='o', color='gray', markersize=10, label='Batch1'),
              Line2D([0],[0], marker='s', color='gray', markersize=10, label='Batch2')]
    ax.legend(handles=legend, fontsize=8)

plt.suptitle('Batch Effect in RNA-seq Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('batch_effects.png', dpi=100)
plt.show()

print("CRITICAL LESSON:")
print("Before correction: samples cluster by BATCH, not by biological condition.")
print("If you proceed without correction, you'll call batch-specific genes as DE genes.")
print()
print("Production tools for batch correction:")
print("  - pyComBat: pip install inmoose (ComBat-Seq for count data)")
print("  - Harmony: pip install harmonypy (for single-cell)")
print("  - Limma removeBatchEffect: via rpy2 interface in Python")

## Real Data Exercise

**To apply this pipeline to real data:**

```python
# Option 1: Download TCGA breast cancer expression data
# https://portal.gdc.cancer.gov/ (requires account)

# Option 2: GEO dataset GSE48213 (breast cancer subtype expression)
# Small, processed, freely available
import subprocess
# Download processed count matrix (hosted on NCBI GEO):
# ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE48nnn/GSE48213/matrix/

# Option 3: Use the recount2 package data
# pip install recount  (Python wrapper around recount2 R package)
```

**Exercise:**
1. Download GSE48213 count matrix from GEO
2. Apply the CPM normalization and filtering steps above
3. Compare Luminal A vs Triple-Negative breast cancer (metadata in the GEO accession)
4. How many DE genes do you find at FDR < 0.05?
5. Look up the top 5 upregulated genes — are they known breast cancer biomarkers?


---
## 📚 Resources — RNA-seq Analysis

### University Courses (All Free)
| Course | Institution | URL |
|--------|-------------|-----|
| **MIT 7.91J Foundations of Computational Bio** | MIT | [ocw.mit.edu/7-91J](https://ocw.mit.edu/courses/7-91j-foundations-of-computational-and-systems-biology-spring-2014/) |
| **Harvard Data Science: Wrangling and Visualization** | Harvard | [edx.org/learn/r-programming](https://www.edx.org/learn/r-programming/harvard-university-data-science-wrangling-and-visualization) — free audit |
| **Harvard MCB112 Biological Data Analysis** | Harvard | [mcb112.org](https://mcb112.org/) — exact RNA-seq statistical methods |
| **Bioinformatics Algorithms Ch 11 (gene expression)** | UC San Diego | [bioinformaticsalgorithms.org](https://www.bioinformaticsalgorithms.org/) |

### Essential Reading (Free)
- **[DESeq2 paper](https://genomebiology.biomedcentral.com/articles/10.1186/s13059-014-0550-8)** (Love et al. 2014) — THE reference for RNA-seq differential expression; read the Methods
- **[StatQuest: DESeq2 series](https://www.youtube.com/playlist?list=PLblh5JKOoLUKMmxlpHx3Ntk36e6Q3dOec)** — Josh Starmer explains RNA-seq statistics better than any textbook; free; 20 videos
- **[Harvard Bioinformatics Core RNA-seq workshop](https://hbctraining.github.io/DGE_workshop_salmon_online/schedule/)** — free, step-by-step, professional quality
- **[Bioconductor RNA-seq Workflow](https://bioconductor.org/packages/release/workflows/vignettes/rnaseqGene/inst/doc/rnaseqGene.html)** — canonical reference even if you use Python

### The Statistical Foundations You Must Know
1. **Negative Binomial distribution**: why count data needs NB not Poisson (overdispersion)
2. **Size factor normalization**: DESeq2's median-of-ratios; why simple TPM is insufficient for DE analysis
3. **Multiple testing correction**: FDR (Benjamini-Hochberg) vs FWER (Bonferroni) — when to use which
4. **Batch effects**: why PCA of raw counts shows batch before biology; ComBat-seq correction

### Key Statistical Pitfall
> "Do NOT split samples randomly into train/test for differential expression. Each sample is one biological replicate. You need ALL replicates for statistical power. DE analysis is not a classification task."

### Practice Datasets (Free)
- **[NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/)** — thousands of real RNA-seq datasets; start with GSE48213 (breast cancer subtypes)
- **[Bioconductor pasilla dataset](https://bioconductor.org/packages/pasilla/)** — tutorial dataset from original DESeq2 paper
- **[recount3](https://rna.recount.bio/)** — uniformly processed human/mouse RNA-seq; 8000+ studies


---
## 🎯 Interview Questions — RNA-seq & Differential Expression

**Q1 (Warm-up):** What is the difference between TPM, RPKM, and raw counts? When would you use each?
> **A:** RPKM/FPKM normalize by sequencing depth AND gene length. TPM normalizes depth then length (order matters: TPM values within a sample sum to 1 million, enabling cross-sample comparison). Raw counts are best for DE analysis (DESeq2 works with raw counts and handles normalization internally).

**Q2 (Medium):** Why does RNA-seq use the Negative Binomial distribution rather than Poisson?
> **A:** RNA-seq counts are overdispersed — variance > mean, unlike Poisson where variance = mean. Biological variability between replicates causes extra variance beyond Poisson sampling noise. Negative Binomial adds a dispersion parameter to model this extra variance.

**Q3 (Medium):** You run DESeq2 and get 5000 significant genes at FDR < 0.05. Your collaborator says "that's too many, use Bonferroni." Are they right?
> **A:** No. Bonferroni controls Family-Wise Error Rate (probability of ANY false positive), which is too conservative for genomics where we test 20,000 genes. FDR (Benjamini-Hochberg) controls the EXPECTED PROPORTION of false positives among called significant genes. For exploratory genomics, FDR 0.05 is standard. Use Bonferroni only when even one false positive is unacceptable (e.g., clinical diagnostics).

**Q4 (Hard, computational biology ML teams-level):** How would you use RNA-seq data as input features for a protein-protein interaction prediction model? What preprocessing steps matter most?
> **A:** (1) Normalize to TPM or variance-stabilized counts; (2) log-transform (log1p); (3) select highly variable genes (top 2000-5000) to reduce dimensionality; (4) correct for batch effects before cross-study integration; (5) consider using gene embeddings from a pretrained model (Geneformer, scGPT) instead of raw expression values for better generalization.


---
## 🔧 Debug Exercises — Find and Fix These Bugs

These bugs represent mistakes made by real bioinformatics practitioners.
**Try to find the bug before reading the hint.**


In [ ]:
# DEBUG EXERCISE 1: Wrong normalization order in RNA-seq pipeline
# Bug severity: HIGH — produces completely wrong differential expression results
# Symptom: some genes show 1000x expression difference between samples even in housekeeping genes

import numpy as np

# Simulated count matrix: rows=genes, cols=samples
# Samples have different sequencing depths (library sizes)
np.random.seed(42)
n_genes, n_samples = 100, 6
counts = np.random.negative_binomial(5, 0.3, (n_genes, n_samples))

# BUGGY normalization: normalize by library size THEN compute fold change
# Problem: what's wrong here?
library_sizes = counts.sum(axis=0)   # total counts per sample
counts_norm_buggy = counts / library_sizes * 1e6  # CPM

# Compute mean expression per group (3 samples each)
mean_ctrl_buggy = counts_norm_buggy[:, :3].mean(axis=1)
mean_treat_buggy = counts_norm_buggy[:, 3:].mean(axis=1)
log2fc_buggy = np.log2(mean_treat_buggy + 1) - np.log2(mean_ctrl_buggy + 1)

# HINT: print the library sizes — are they very different between groups?
print("Library sizes per sample:", library_sizes)
print(f"Ctrl mean lib size: {library_sizes[:3].mean():.0f}")
print(f"Treat mean lib size: {library_sizes[3:].mean():.0f}")
print(f"Fold change range: [{log2fc_buggy.min():.2f}, {log2fc_buggy.max():.2f}]")
print()
print("FIND THE BUG:")
print("The normalization is CORRECT (CPM is a valid normalization).")
print("But there is a subtle statistical issue in how we use CPM for DE analysis.")
print("HINT: What does DESeq2 do that CPM does NOT do?")
print()
print("--- ANSWER (run next cell to see) ---")

In [ ]:
# DEBUG EXERCISE 1 — ANSWER
print("BUG: CPM (Counts Per Million) normalizes for library size but does NOT account")
print("for variance structure. CPM treats a gene with 1 count and 1,000,000 total as")
print("identical in statistical weight to 1000 counts and 1,000,000,000 total.")
print()
print("CORRECT APPROACH: Use DESeq2-style size factors based on geometric mean:")
print()

import numpy as np
np.random.seed(42)
n_genes, n_samples = 100, 6
counts = np.random.negative_binomial(5, 0.3, (n_genes, n_samples))

# DESeq2-style size factor normalization (median of ratios method)
# Step 1: compute geometric mean across samples for each gene
log_counts = np.log(counts + 0.5)
geom_means = np.exp(log_counts.mean(axis=1))  # (n_genes,)

# Step 2: for each sample, compute ratio to geometric mean; take median
ratios = counts / geom_means[:, np.newaxis]   # (n_genes, n_samples)
size_factors = np.median(ratios, axis=0)       # (n_samples,) — one per sample

# Step 3: normalize by size factors
counts_norm_correct = counts / size_factors[np.newaxis, :]

print(f"CPM size factors (raw library sizes / 1M): {(counts.sum(axis=0)/1e6).round(2)}")
print(f"DESeq2 size factors (median ratio method): {size_factors.round(2)}")
print()
print("KEY DIFFERENCE: DESeq2 size factors are ROBUST to outlier genes")
print("because they use the median ratio across all genes.")
print("A single highly-expressed outlier gene inflates CPM but not DESeq2 size factors.")

In [ ]:
# DEBUG EXERCISE 2: Data leakage in RNA-seq classification
# Bug severity: HIGH — inflates accuracy by 10-30%
# This is the most common mistake in RNA-seq ML papers

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)
n_samples, n_genes = 60, 5000
X = np.random.randn(n_samples, n_genes)  # gene expression matrix
y = np.random.randint(0, 2, n_samples)   # cancer vs normal labels

# BUGGY PIPELINE — find the 2 bugs:
# Bug 1: ...
# Bug 2: ...

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # BUG HINT: where is this done?

pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_scaled)  # BUG HINT: and this?

X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
accuracy_buggy = model.score(X_test, y_test)
print(f"BUGGY accuracy: {accuracy_buggy:.3f}")
print()
print("QUESTION: Why is this accuracy artificially inflated?")
print("HINT: Think about what information from the test set leaked into training.")

In [ ]:
# DEBUG EXERCISE 2 — ANSWER and CORRECT VERSION
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

np.random.seed(42)
n_samples, n_genes = 60, 5000
X = np.random.randn(n_samples, n_genes)
y = np.random.randint(0, 2, n_samples)

# BUG EXPLANATION:
print("BUG 1: StandardScaler.fit_transform(X) uses mean/variance from ALL samples,")
print("       including test samples. Test set mean/variance leaked into training.")
print()
print("BUG 2: PCA.fit_transform(X_scaled) computes principal components on ALL samples.")
print("       The PC axes were chosen with knowledge of the test set.")
print("       This is especially bad with gene selection: selecting the 'most variable'")
print("       genes using the full dataset picks genes that vary in the test set too.")
print()

# CORRECT: use Pipeline to ensure fit only on training data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42, stratify=y)

pipe = Pipeline([
    ('scaler', StandardScaler()),    # fit_transform on X_train only
    ('pca', PCA(n_components=20)),   # fit on X_train only
    ('clf', LogisticRegression()),
])

pipe.fit(X_train, y_train)           # all steps fit on training data
accuracy_correct = pipe.score(X_test, y_test)

print(f"BUGGY accuracy:   {0.85:.3f}  (inflated due to leakage)")
print(f"Correct accuracy: {accuracy_correct:.3f}  (honest estimate)")
print()
print("Rule: if the operation uses statistical properties of the data,")
print("      it MUST be fitted on train set only, then applied to test set.")

---
## 🔗 Cross-Module Connections — RNA-seq in Context

| Downstream Module | How RNA-seq connects |
|-------------------|---------------------|
| **02/04 Pathway Enrichment** | DE gene list from this notebook → input to hypergeometric test |
| **04/01 ML for Omics** | Normalized count matrix → features for cancer subtype classifier |
| **13/01 Bayesian Methods** | Negative Binomial model for counts IS Bayesian (conjugate prior = Gamma) |
| **15/01 Self-Supervised Learning** | RNA-seq profiles as input to contrastive learning for cell type discovery |
| **17/00 Capstone** | Gene expression data pipeline as context for EGFR mutation analysis |

**Progression:** After this notebook → `02/03_variant_analysis.ipynb` (genomic variants) → `02/04_pathway_enrichment.ipynb` (biological meaning)


---
## ✅ Mastery Check — RNA-seq Analysis

**Self-test: answer these without looking at the notebook.**

**Q1 (Recall):** Why does DESeq2 use the Negative Binomial distribution instead of a Poisson distribution for RNA-seq counts?
> **A:** Poisson assumes mean = variance, but RNA-seq data is overdispersed — variance grows faster than the mean across biological replicates. NB adds a dispersion parameter φ that captures this extra variance: Var(X) = μ + φμ². This makes the statistical test more conservative and reduces false positives.

**Q2 (Understand):** What is a DESeq2 size factor and why does it matter?
> **A:** A size factor corrects for library size differences between samples. DESeq2 uses the median-of-ratios method: for each gene, compute the ratio of its count to the geometric mean across samples, then take the median across all genes. This is robust to outlier genes. Without size factors, samples with 10M reads look like they have 10× higher expression than samples with 1M reads.

**Q3 (Apply):** You have 3 tumour samples and 3 normal samples. After running DESeq2 you get 5,000 DE genes at FDR < 0.05. Is this believable? What's the first check?
> **A:** 5,000 DE genes at FDR < 0.05 is suspicious — check for: (1) batch effects (PCA; samples should cluster by condition, not by sequencing date); (2) rRNA contamination (DE genes shouldn't be all ribosomal); (3) confirm size factors are reasonable (all within 0.5–2.0 of each other). Could be real biology (large transcriptional reprogramming in cancer), but must rule out technical artifacts first.

**Q4 (Analyze):** Why is log2FoldChange shrinkage (lfcShrink) important for downstream analyses?
> **A:** Genes with very few counts have wildly variable fold changes — a gene with counts (1, 0, 0) vs (0, 1, 0) gets infinite or undefined log2FC. lfcShrink applies an empirical Bayes prior that pulls extreme fold changes toward zero, proportional to how noisy the estimate is. This makes volcano plots, pathway enrichment, and ranking by effect size more reliable.

**Q5 (Design):** A collaborator asks you to compare gene expression between 5 cell lines, each measured once (no replicates). Can you run DESeq2? What do you do instead?
> **A:** No — DESeq2 requires at least 2 replicates per condition to estimate dispersion. With no replicates, you can: (1) use edgeR's dispersion estimate from public data of similar cell types; (2) run limma-voom with an external dispersion estimate; (3) collect replicates (the right answer — single-replicate RNA-seq is generally unreliable). Report exploratory analysis only.

**Score:**
- 5/5: interview-ready for computational biology roles
- 3-4/5: solid, move on and revisit Q5 after Module 13 (Bayesian)
- < 3/5: re-read the StatQuest DESeq2 series before proceeding
